# Reddit Data Collection: BA/DA Interview Assessment Study

## 1. Data Collection Plan

### Population
Reddit posts from r/datascience and r/analytics discussing BA, DA, and analytics-related interview experiences.

### Search Terms
business analyst interview，data analyst interview，analytics internship interview，SQL interview，case interview，ChatGPT interview，AI tools interview，Copilot interview，GPT interview，LLM interview

### Time Window
January 2023 - September 2026

## 2. Environment Setup

This notebook uses the following Python packages:

- praw: access Reddit API
- python-dotenv: load API credentials from .env file

## 3. Reddit API Connection

In [42]:
from dotenv import load_dotenv
import os

load_dotenv()

client_id = os.getenv("REDDIT_CLIENT_ID")
client_secret = os.getenv("REDDIT_CLIENT_SECRET")
user_agent = os.getenv("REDDIT_USER_AGENT")


In [43]:
import praw

reddit = praw.Reddit(
    client_id=client_id,
    client_secret=client_secret,
    user_agent=user_agent
)

print(reddit.read_only)

True


In [ ]:
## Test
for submission in reddit.subreddit("dataanalysis").hot(limit=5):
    print(submission.title)

Announcing DataAnalysisCareers
Web scraping
UCBLogo Reading and transforming CSV files
Pls help
Reconciliation & Edge Cases Focus


## 4. Post Collection

In [62]:
import pandas as pd
from datetime import datetime, timezone

##keywords

keywords = [
    "business analyst interview experience",
    "data analyst interview experience",
    "analytics internship interview",
    "SQL interview",
    "case study interview",
    "ChatGPT interview",
    "AI tools interview",
    "Copilot interview",
    "GPT interview",
    "LLM interview"
]

##Window time
start_date = datetime(2023, 1, 1, tzinfo=timezone.utc)
end_date = datetime(2026, 9, 22, tzinfo=timezone.utc)

start_timestamp = start_date.timestamp()
end_timestamp = end_date.timestamp()


posts = []

for keyword in keywords:

    for subreddit_name in ["analytics", "datascience"]:

        subreddit = reddit.subreddit(subreddit_name)

        for post in subreddit.search(keyword, limit=100):

            if post.created_utc < start_timestamp:
                continue

            if post.created_utc > end_timestamp:
                continue


            posts.append({
                "keyword": keyword,
                "subreddit": subreddit_name,
                "title": post.title,
                "text": post.selftext,
                "date": datetime.fromtimestamp(post.created_utc,tz=timezone.utc).strftime("%Y-%m-%d"),
                "score": post.score,
                "url": post.url
            })


df = pd.DataFrame(posts)


df = df.drop_duplicates(subset=["url"])


print("Unique posts:", df.shape)

df.head()

Unique posts: (582, 7)


,keyword,subreddit,title,text,date,score,url
0,business analyst interview experience,analytics,3+ YOE Data Analyst barely getting interview c...,"Looking for honest feedback, especially from r...",2026-09-18,32,https://www.reddit.com/r/analytics/comments/1w...
1,business analyst interview experience,analytics,1.7 years of Data Analyst experience + 2.5-yea...,"Hi everyone,\n\nI have **1.7 years of experien...",2026-07-29,60,https://www.reddit.com/r/analytics/comments/1v...
2,business analyst interview experience,analytics,Data Analyst interview process — how many roun...,"Hi all,\n\nI’m preparing for switch to Data An...",2025-08-18,13,https://www.reddit.com/r/analytics/comments/1m...
3,business analyst interview experience,analytics,Recent interviews experience,\nI’m seeking some guidance regarding my job s...,2025-03-04,10,https://www.reddit.com/r/analytics/comments/1j...
4,business analyst interview experience,analytics,Im looking for a professional in the field to ...,"Hi, im lookkng to start a series where i tackl...",2025-03-02,8,https://www.reddit.com/r/analytics/comments/1j...


In [63]:
df.to_csv("reddit_interview_pilot.csv", index=False)

## 6. Collection Statistics

In [64]:
print("Total records:", len(df))
print("Subreddits:")
print(df["subreddit"].value_counts())

print("\nDate range:")
print(df["date"].min(), "to", df["date"].max())

Total records: 582
Subreddits:
subreddit
analytics      357
datascience    225
Name: count, dtype: int64

Date range:
2023-01-03 to 2026-09-21
